# CAB420 Assignment 2

## Phase 1: Data Pipeline

Builds `shared_data.csv` — the shared dataset for all three group methods.
Top 15 Enron authors, cleaned email bodies, stratified 70/15/15 split.

In [8]:
import os
import re
import email
import pathlib
import pandas as pd
from sklearn.model_selection import train_test_split

DATA_DIR: pathlib.Path = pathlib.Path("enron_mail_20150507/maildir")

In [9]:
PERSONAL_ACCOUNT_PATTERN: re.Pattern = re.compile(r"^[a-z]+(-[a-z0-9]+)+$")
MIN_EMAILS: int = 200
TOP_N: int = 15  # D-05


def count_emails_per_author(data_dir: pathlib.Path) -> dict[str, int]:
    """
    Walk data_dir one level deep. For each subdirectory whose name matches
    PERSONAL_ACCOUNT_PATTERN, count all files recursively (each file is one email).
    Uses os.walk instead of rglob — Enron filenames end with '.' (e.g. '1.')
    and Windows pathlib.rglob silently skips trailing-dot filenames.
    Returns dict mapping author_dir_name -> file count.
    """
    counts: dict[str, int] = {}
    for entry in sorted(data_dir.iterdir()):
        if not entry.is_dir():
            continue
        if not PERSONAL_ACCOUNT_PATTERN.match(entry.name):
            continue
        file_count: int = sum(len(files) for _, _, files in os.walk(str(entry)))
        counts[entry.name] = file_count
    return counts


def select_top_authors(counts: dict[str, int], min_emails: int, top_n: int) -> list[str]:
    """
    Filter authors below min_emails, sort descending by count, return top_n names.
    Per D-06: count-based selection after 200-email minimum filter.
    """
    eligible: dict[str, int] = {a: c for a, c in counts.items() if c >= min_emails}
    sorted_authors: list[str] = sorted(eligible, key=lambda a: eligible[a], reverse=True)
    return sorted_authors[:top_n]


raw_counts: dict[str, int] = count_emails_per_author(DATA_DIR)
selected_authors: list[str] = select_top_authors(raw_counts, MIN_EMAILS, TOP_N)

print(f"Authors with >= {MIN_EMAILS} emails: {len([c for c in raw_counts.values() if c >= MIN_EMAILS])}")
print(f"Selected top {TOP_N}: {selected_authors}")

Authors with >= 200 emails: 148
Selected top 15: ['kaminski-v', 'dasovich-j', 'kean-s', 'mann-k', 'jones-t', 'shackleton-s', 'taylor-m', 'farmer-d', 'germany-c', 'beck-s', 'symes-k', 'nemec-g', 'scott-s', 'rogers-b', 'bass-e']


In [10]:
def _open_path(file_path: pathlib.Path):
    """
    Return a file-like object for file_path.
    On Windows, Enron filenames end with '.' (e.g. '1.'). The Win32 open() call
    silently strips trailing dots, causing FileNotFoundError. The extended-path
    prefix (\\\\?\\) bypasses Win32 normalisation and opens the file correctly.
    """
    if os.name == "nt":
        path_str = "\\\\?\\" + str(file_path.absolute()).replace("/", "\\")
    else:
        path_str = str(file_path)
    return open(path_str, "r", encoding="utf-8", errors="replace")


def parse_email_body(file_path: pathlib.Path) -> str | None:
    """
    Open file_path, parse with Python stdlib email module (D-13), return payload string.
    Returns None if the payload is not a string (e.g. multipart without text/plain part).
    Headers (To/From/Subject/Date/etc.) are stripped automatically by msg.get_payload().
    """
    with _open_path(file_path) as fh:
        msg: email.message.Message = email.message_from_file(fh)
    payload: str | bytes | list = msg.get_payload()
    if isinstance(payload, str):
        return payload
    if isinstance(payload, list):
        for part in payload:
            if part.get_content_type() == "text/plain":
                part_payload = part.get_payload()
                if isinstance(part_payload, str):
                    return part_payload
    return None

In [11]:
FORWARDED_MARKER: str = "-----Original Message-----"

SIGNATURE_TRIGGER_PATTERNS: list[re.Pattern] = [
    re.compile(r"\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}"),                           # phone numbers
    re.compile(r"Director|Manager|Vice President|VP|Analyst|President|Officer|Associate", re.IGNORECASE),  # job titles (D-09)
    re.compile(r"Enron|ECT|ENA|Corp", re.IGNORECASE),                            # company names (D-09)
]

URL_PATTERN: re.Pattern = re.compile(r"https?://\S+|www\.\S+")           # D-11
EMAIL_PATTERN: re.Pattern = re.compile(r"[\w.+-]+@[\w-]+\.[a-z]{2,}")    # D-12


def strip_forwarded(body: str) -> str:
    """
    Remove everything at and after the first FORWARDED_MARKER (D-14).
    If marker is absent, return body unchanged.
    """
    idx: int = body.find(FORWARDED_MARKER)
    if idx == -1:
        return body
    return body[:idx]


def strip_signature(body: str) -> str:
    """
    Examine the last 5 lines. If >= 2 of them match any SIGNATURE_TRIGGER_PATTERNS,
    strip all trailing lines that match any pattern (D-08, D-09).
    Strips from the bottom up — stops at the first non-matching line.
    """
    lines: list[str] = body.rstrip("\n").splitlines()
    tail: list[str] = lines[-5:]
    match_count: int = sum(
        1 for line in tail
        if any(pat.search(line) for pat in SIGNATURE_TRIGGER_PATTERNS)
    )
    if match_count < 2:
        return body
    while lines and any(pat.search(lines[-1]) for pat in SIGNATURE_TRIGGER_PATTERNS):
        lines.pop()
    return "\n".join(lines)


def normalise(body: str) -> str:
    """
    Apply in order (D-10, D-11, D-12):
    1. Replace URLs with URL_TOKEN
    2. Replace email addresses with EMAIL_TOKEN
    3. Lowercase the entire body
    Punctuation is preserved — Alex's SVM uses char n-gram TF-IDF (P8).
    """
    body = URL_PATTERN.sub("URL_TOKEN", body)
    body = EMAIL_PATTERN.sub("EMAIL_TOKEN", body)
    body = body.lower()
    return body


def is_long_enough(body: str, min_words: int = 20) -> bool:
    """
    Return True if body contains at least min_words whitespace-separated tokens (D-15, P10).
    """
    return len(body.split()) >= min_words

In [12]:
def process_author_emails(
    author_name: str,
    author_dir: pathlib.Path,
) -> list[dict[str, str]]:
    """
    Walk author_dir recursively. For each file:
      1. parse_email_body  -> raw body string (skip if None)
      2. strip_forwarded   -> remove replied/forwarded content
      3. strip_signature   -> remove trailing signature block
      4. normalise         -> lowercase, URL_TOKEN, EMAIL_TOKEN
      5. is_long_enough    -> skip if < 20 words
    Uses os.walk instead of rglob — Enron filenames end with '.' (e.g. '1.')
    and Windows pathlib.rglob silently skips trailing-dot filenames.
    Returns list of dicts with keys: author_label (str), cleaned_body (str).
    email_id is assigned by the caller.
    """
    records: list[dict[str, str]] = []
    for dirpath, _, filenames in os.walk(str(author_dir)):
        for filename in sorted(filenames):
            file_path: pathlib.Path = pathlib.Path(dirpath) / filename
            raw_body: str | None = parse_email_body(file_path)
            if raw_body is None:
                continue
            body: str = strip_forwarded(raw_body)
            body = strip_signature(body)
            body = normalise(body)
            if not is_long_enough(body):
                continue
            records.append({"author_label": author_name, "cleaned_body": body})
    return records


all_records: list[dict[str, str]] = []
email_id_counter: int = 0

for author in selected_authors:
    author_dir: pathlib.Path = DATA_DIR / author
    author_records: list[dict[str, str]] = process_author_emails(author, author_dir)
    for record in author_records:
        record["email_id"] = f"email_{email_id_counter:06d}"
        email_id_counter += 1
        all_records.append(record)
    print(f"{author}: {len(author_records)} emails after filtering")

print(f"\nTotal records: {len(all_records)}")

kaminski-v: 25343 emails after filtering
dasovich-j: 23877 emails after filtering
kean-s: 22414 emails after filtering
mann-k: 20343 emails after filtering
jones-t: 17321 emails after filtering
shackleton-s: 16921 emails after filtering
taylor-m: 12331 emails after filtering
farmer-d: 10755 emails after filtering
germany-c: 9946 emails after filtering
beck-s: 10766 emails after filtering
symes-k: 9664 emails after filtering
nemec-g: 8774 emails after filtering
scott-s: 7158 emails after filtering
rogers-b: 6383 emails after filtering
bass-e: 6264 emails after filtering

Total records: 208260


In [13]:
df: pd.DataFrame = pd.DataFrame(all_records, columns=["email_id", "author_label", "cleaned_body"])

assert df["email_id"].is_unique, "Duplicate email_id values detected — fix counter logic"
assert df["author_label"].nunique() == TOP_N, (
    f"Expected {TOP_N} authors, got {df['author_label'].nunique()}"
)

print("Per-author email counts:")
print(df["author_label"].value_counts().to_string())
print(f"\nTotal rows: {len(df)}")

Per-author email counts:
author_label
kaminski-v      25343
dasovich-j      23877
kean-s          22414
mann-k          20343
jones-t         17321
shackleton-s    16921
taylor-m        12331
beck-s          10766
farmer-d        10755
germany-c        9946
symes-k          9664
nemec-g          8774
scott-s          7158
rogers-b         6383
bass-e           6264

Total rows: 208260


In [14]:
def assign_splits(
    df: pd.DataFrame,
    random_state: int,
) -> pd.DataFrame:
    """
    Split df into train (70%), val (15%), test (15%) using stratified sampling.
    Returns df with a new 'split' column added (values: 'train', 'val', 'test').
    random_state is explicit — no default (per project code style rules).
    """
    train_idx, temp_idx = train_test_split(
        df.index,
        test_size=0.30,
        stratify=df["author_label"],
        random_state=random_state,
    )
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,   # 50% of the 30% temp = 15% of total
        stratify=df.loc[temp_idx, "author_label"],
        random_state=random_state,
    )
    df_out: pd.DataFrame = df.copy()
    df_out["split"] = "train"
    df_out.loc[val_idx, "split"] = "val"
    df_out.loc[test_idx, "split"] = "test"
    return df_out


df_split: pd.DataFrame = assign_splits(df, random_state=42)

split_counts: pd.Series = df_split["split"].value_counts()
total: int = len(df_split)
print(f"train: {split_counts['train']} ({split_counts['train']/total:.1%})")
print(f"val:   {split_counts['val']}   ({split_counts['val']/total:.1%})")
print(f"test:  {split_counts['test']}  ({split_counts['test']/total:.1%})")

for author in df_split["author_label"].unique():
    author_splits: set[str] = set(df_split.loc[df_split["author_label"] == author, "split"])
    missing: set[str] = {"train", "val", "test"} - author_splits
    assert not missing, f"Author {author} missing from split(s): {missing}"
print("Stratification check passed — all authors present in all three splits.")

train: 145782 (70.0%)
val:   31239   (15.0%)
test:  31239  (15.0%)
Stratification check passed — all authors present in all three splits.


In [15]:
OUTPUT_PATH: pathlib.Path = pathlib.Path("shared_data.csv")

df_split.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df_split)} rows to {OUTPUT_PATH.resolve()}\n")

summary: pd.DataFrame = (
    df_split.groupby(["author_label", "split"])
    .size()
    .unstack(fill_value=0)
    [["train", "val", "test"]]
)
print("Per-author email counts:")
print(summary.to_string())
print(f"\nTotal rows: {len(df_split)}")

sample_body: str = df_split.loc[df_split["split"] == "train", "cleaned_body"].iloc[0]
print(f"\nSample cleaned body (first 200 chars):\n{sample_body[:200]}")

Saved 208260 rows to F:\Uni Bach\2026\Semester 1\CAB420 Machine Learning\Assignment 2\CAB420-Assignment-2-Groups-116\shared_data.csv

Per-author email counts:
split         train   val  test
author_label                   
bass-e         4385   940   939
beck-s         7536  1615  1615
dasovich-j    16714  3582  3581
farmer-d       7528  1613  1614
germany-c      6962  1492  1492
jones-t       12125  2598  2598
kaminski-v    17740  3801  3802
kean-s        15690  3362  3362
mann-k        14240  3051  3052
nemec-g        6142  1316  1316
rogers-b       4468   958   957
scott-s        5010  1074  1074
shackleton-s  11845  2538  2538
symes-k        6765  1449  1450
taylor-m       8632  1850  1849

Total rows: 208260

Sample cleaned body (first 200 chars):
vince/stinson,

please find below a summary of the presenation given to lenders at the april 
23rd meeting in london.

the key points that emerge are:
phase ii will require commitments of about $700 m


In [16]:
df_verify: pd.DataFrame = pd.read_csv("shared_data.csv")

# Success criterion 1: correct columns
assert list(df_verify.columns) == ["email_id", "author_label", "cleaned_body", "split"], (
    f"Column mismatch: {list(df_verify.columns)}"
)

# Success criterion 2: exactly 15 authors, each with >= 200 emails
author_counts: pd.Series = df_verify["author_label"].value_counts()
assert df_verify["author_label"].nunique() == TOP_N, (
    f"Expected {TOP_N} authors, got {df_verify['author_label'].nunique()}"
)
assert (author_counts >= 200).all(), (
    f"Authors with < 200 emails: {author_counts[author_counts < 200].to_dict()}"
)

# Success criterion 3: no header lines, forwarded markers, or signature triggers in bodies
LEAK_PATTERNS: list[str] = [
    r"^From:", r"^To:", r"^Subject:", r"^Date:",
    r"-----Original Message-----",
]
for pattern in LEAK_PATTERNS:
    matches: pd.Series = df_verify["cleaned_body"].str.contains(pattern, regex=True, na=False)
    assert not matches.any(), (
        f"Pattern '{pattern}' found in {matches.sum()} rows — preprocessing failed"
    )

# Success criterion 4: no body < 20 words
word_counts: pd.Series = df_verify["cleaned_body"].str.split().str.len()
assert (word_counts >= 20).all(), (
    f"{(word_counts < 20).sum()} emails have fewer than 20 words"
)

# Success criterion 5: split proportions correct (within 2% tolerance)
total: int = len(df_verify)
train_pct: float = (df_verify["split"] == "train").sum() / total
val_pct: float   = (df_verify["split"] == "val").sum()   / total
test_pct: float  = (df_verify["split"] == "test").sum()  / total
assert 0.68 <= train_pct <= 0.72, f"Train split {train_pct:.1%} out of expected 70% range"
assert 0.13 <= val_pct   <= 0.17, f"Val split {val_pct:.1%} out of expected 15% range"
assert 0.13 <= test_pct  <= 0.17, f"Test split {test_pct:.1%} out of expected 15% range"

print("All Phase 1 verification checks passed.")
print(f"  Authors: {df_verify['author_label'].nunique()}")
print(f"  Total rows: {total}")
print(f"  Train: {train_pct:.1%}  Val: {val_pct:.1%}  Test: {test_pct:.1%}")

All Phase 1 verification checks passed.
  Authors: 15
  Total rows: 208260
  Train: 70.0%  Val: 15.0%  Test: 15.0%
